# Preguntas de negocio con DataFrames y SparkSQL

## Iniciar Spark

In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
15,application_1763396678263_0021,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
sc

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<SparkContext master=yarn appName=livy-session-15>

## Configuración del entorno

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, upper, count, when, avg, sum as spark_sum,
    year, month, dayofmonth, desc, asc, round as spark_round
)
from pyspark.sql.types import IntegerType, StringType

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Cargar datos de S3

In [4]:
bucket = "lmtorresv-datalake"
path = f"s3a://{bucket}/datasets/covid19/Casos_positivos_de_COVID-19_en_Colombia-100K.csv"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
df_raw = spark.read.csv(
    path,
    header=True,
    inferSchema=True,
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Transformación de datos

### Renombrar columnas

In [6]:
df = df_raw.toDF(
    "fecha_reporte",
    "id_caso",
    "fecha_notificacion",
    "cod_depto",
    "departamento",
    "cod_municipio",
    "ciudad",
    "edad",
    "unidad_edad",
    "sexo",
    "tipo_contagio",
    "ubicacion",
    "estado",
    "cod_pais",
    "pais",
    "recuperado",
    "fecha_sintomas",
    "fecha_muerte",
    "fecha_diagnostico",
    "fecha_recuperacion",
    "tipo_recuperacion",
    "etnia",
    "grupo_etnico"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### Normalizar y transformar datos

In [7]:
# Normalizar texto a mayúsculas
df = df.withColumn("departamento", upper(col("departamento")))
df = df.withColumn("ciudad", upper(col("ciudad")))
df = df.withColumn("sexo", upper(col("sexo")))
df = df.withColumn("estado", upper(col("estado")))
df = df.withColumn("ubicacion", upper(col("ubicacion")))
df = df.withColumn("recuperado", upper(col("recuperado")))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### Cachear DataFrame

In [8]:
df.cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[fecha_reporte: string, id_caso: int, fecha_notificacion: string, cod_depto: int, departamento: string, cod_municipio: int, ciudad: string, edad: int, unidad_edad: int, sexo: string, tipo_contagio: string, ubicacion: string, estado: string, cod_pais: int, pais: string, recuperado: string, fecha_sintomas: string, fecha_muerte: string, fecha_diagnostico: string, fecha_recuperacion: string, tipo_recuperacion: string, etnia: int, grupo_etnico: string]

In [9]:
df.show(1, vertical=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

-RECORD 0-------------------------------
 fecha_reporte      | 6/3/2020 0:00:00  
 id_caso            | 1                 
 fecha_notificacion | 2/3/2020 0:00:00  
 cod_depto          | 11                
 departamento       | BOGOTA            
 cod_municipio      | 11001             
 ciudad             | BOGOTA            
 edad               | 19                
 unidad_edad        | 1                 
 sexo               | F                 
 tipo_contagio      | Importado         
 ubicacion          | CASA              
 estado             | LEVE              
 cod_pais           | 380               
 pais               | ITALIA            
 recuperado         | RECUPERADO        
 fecha_sintomas     | 27/2/2020 0:00:00 
 fecha_muerte       | NULL              
 fecha_diagnostico  | 6/3/2020 0:00:00  
 fecha_recuperacion | 13/3/2020 0:00:00 
 tipo_recuperacion  | PCR               
 etnia              | 6                 
 grupo_etnico       | NULL              
only showing top

## Análisis con DataFrames

### 1. Top 10 departamentos con más casos

In [10]:
top_departamentos = (
    df.groupBy("departamento")
    .agg(count("*").alias("total_casos"))
    .orderBy(desc("total_casos"))
    .limit(10)
)

top_departamentos.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-----------+
|departamento|total_casos|
+------------+-----------+
|BOGOTA      |30016      |
|BARRANQUILLA|13065      |
|ATLANTICO   |10994      |
|VALLE       |10404      |
|CARTAGENA   |8333       |
|ANTIOQUIA   |4554       |
|NARIÑO      |3520       |
|CUNDINAMARCA|2827       |
|AMAZONAS    |2317       |
|CHOCO       |1636       |
+------------+-----------+

### 2. Top 10 ciudades con más casos

In [11]:
top_ciudades = (
    df.groupBy("ciudad", "departamento")
    .agg(count("*").alias("total_casos"))
    .orderBy(desc("total_casos"))
    .limit(10)
)

top_ciudades.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+------------+-----------+
|ciudad      |departamento|total_casos|
+------------+------------+-----------+
|BOGOTA      |BOGOTA      |30016      |
|BARRANQUILLA|BARRANQUILLA|13065      |
|CARTAGENA   |CARTAGENA   |8333       |
|CALI        |VALLE       |7747       |
|SOLEDAD     |ATLANTICO   |6233       |
|LETICIA     |AMAZONAS    |2194       |
|MEDELLIN    |ANTIOQUIA   |2137       |
|TUMACO      |NARIÑO      |1501       |
|BUENAVENTURA|VALLE       |1453       |
|QUIBDO      |CHOCO       |1367       |
+------------+------------+-----------+

### 3. Top 10 días con más casos

In [13]:
top_dias = (
    df.groupBy("fecha_notificacion")
    .agg(count("*").alias("total_casos"))
    .orderBy(desc("total_casos"))
    .limit(10)
)

top_dias.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+-----------+
|fecha_notificacion|total_casos|
+------------------+-----------+
|18/6/2020 0:00:00 |3477       |
|19/6/2020 0:00:00 |3328       |
|17/6/2020 0:00:00 |3318       |
|16/6/2020 0:00:00 |3232       |
|23/6/2020 0:00:00 |3230       |
|11/6/2020 0:00:00 |2747       |
|20/6/2020 0:00:00 |2684       |
|12/6/2020 0:00:00 |2679       |
|10/6/2020 0:00:00 |2650       |
|24/6/2020 0:00:00 |2599       |
+------------------+-----------+

### 4. Distribución de casos por edad

In [14]:
# Crear rangos de edad
df_edad = df.withColumn(
    "rango_edad",
    when(col("edad") < 18, "0-17")
    .when((col("edad") >= 18) & (col("edad") < 30), "18-29")
    .when((col("edad") >= 30) & (col("edad") < 40), "30-39")
    .when((col("edad") >= 40) & (col("edad") < 50), "40-49")
    .when((col("edad") >= 50) & (col("edad") < 60), "50-59")
    .when((col("edad") >= 60) & (col("edad") < 70), "60-69")
    .when((col("edad") >= 70) & (col("edad") < 80), "70-79")
    .when(col("edad") >= 80, "80+")
    .otherwise("Sin dato")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
distribucion_edad = (
    df_edad.groupBy("rango_edad")
    .agg(count("*").alias("total_casos"))
    .orderBy("rango_edad")
)

distribucion_edad.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----------+
|rango_edad|total_casos|
+----------+-----------+
|0-17      |8587       |
|18-29     |24664      |
|30-39     |23029      |
|40-49     |15828      |
|50-59     |12857      |
|60-69     |7955       |
|70-79     |4362       |
|80+       |2718       |
+----------+-----------+

In [16]:
# Estadísticas de edad
df.select(
    spark_round(avg("edad"), 1).alias("edad_promedio"),
    spark_round(spark_sum("edad") / count("*"), 1).alias("edad_media")
).show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+----------+
|edad_promedio|edad_media|
+-------------+----------+
|         39.3|      39.3|
+-------------+----------+

### 5. Tasa de letalidad por departamento

Pregunta de negocio adicional: ¿Cuál es la tasa de letalidad (porcentaje de fallecidos sobre total de casos) por departamento?

In [20]:
letalidad_departamento = (
    df.groupBy("departamento")
    .agg(
        count("*").alias("total_casos"),
        spark_sum(when(col("estado") == "FALLECIDO", 1).otherwise(0)).alias("fallecidos")
    )
    .withColumn(
        "tasa_letalidad_%",
        spark_round((col("fallecidos") / col("total_casos")) * 100, 2)
    )
    .orderBy(desc("tasa_letalidad_%"))
    .limit(15)
)

letalidad_departamento.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-----------+----------+----------------+
|departamento   |total_casos|fallecidos|tasa_letalidad_%|
+---------------+-----------+----------+----------------+
|PUTUMAYO       |28         |7         |25.0            |
|MAGDALENA      |869        |100       |11.51           |
|CORDOBA        |807        |89        |11.03           |
|NORTE SANTANDER|343        |28        |8.16            |
|SUCRE          |1317       |103       |7.82            |
|GUAJIRA        |470        |34        |7.23            |
|GUAINIA        |14         |1         |7.14            |
|BARRANQUILLA   |13065      |926       |7.09            |
|STA MARTA D.E. |1022       |69        |6.75            |
|ATLANTICO      |10994      |662       |6.02            |
|BOLIVAR        |998        |60        |6.01            |
|CAUCA          |402        |23        |5.72            |
|CHOCO          |1636       |93        |5.68            |
|VALLE          |10404      |565       |5.43            |
|CARTAGENA    

## Análisis con SparkSQL

### Registrar tabla temporal

In [21]:
df.createOrReplaceTempView("covid")
df_edad.createOrReplaceTempView("covid_edad")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 1. Top 10 departamentos con más casos (SQL)

In [22]:
sql_departamentos = spark.sql("""
    SELECT 
        departamento,
        COUNT(*) as total_casos
    FROM covid
    GROUP BY departamento
    ORDER BY total_casos DESC
    LIMIT 10
""")

sql_departamentos.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-----------+
|departamento|total_casos|
+------------+-----------+
|BOGOTA      |30016      |
|BARRANQUILLA|13065      |
|ATLANTICO   |10994      |
|VALLE       |10404      |
|CARTAGENA   |8333       |
|ANTIOQUIA   |4554       |
|NARIÑO      |3520       |
|CUNDINAMARCA|2827       |
|AMAZONAS    |2317       |
|CHOCO       |1636       |
+------------+-----------+

### 2. Top 10 ciudades con más casos (SQL)

In [23]:
sql_ciudades = spark.sql("""
    SELECT 
        ciudad,
        departamento,
        COUNT(*) as total_casos
    FROM covid
    GROUP BY ciudad, departamento
    ORDER BY total_casos DESC
    LIMIT 10
""")

sql_ciudades.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+------------+-----------+
|ciudad      |departamento|total_casos|
+------------+------------+-----------+
|BOGOTA      |BOGOTA      |30016      |
|BARRANQUILLA|BARRANQUILLA|13065      |
|CARTAGENA   |CARTAGENA   |8333       |
|CALI        |VALLE       |7747       |
|SOLEDAD     |ATLANTICO   |6233       |
|LETICIA     |AMAZONAS    |2194       |
|MEDELLIN    |ANTIOQUIA   |2137       |
|TUMACO      |NARIÑO      |1501       |
|BUENAVENTURA|VALLE       |1453       |
|QUIBDO      |CHOCO       |1367       |
+------------+------------+-----------+

### 3. Top 10 días con más casos (SQL)

In [24]:
sql_dias = spark.sql("""
    SELECT 
        fecha_notificacion,
        COUNT(*) as total_casos
    FROM covid
    WHERE fecha_notificacion IS NOT NULL
    GROUP BY fecha_notificacion
    ORDER BY total_casos DESC
    LIMIT 10
""")

sql_dias.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+-----------+
|fecha_notificacion|total_casos|
+------------------+-----------+
|18/6/2020 0:00:00 |3477       |
|19/6/2020 0:00:00 |3328       |
|17/6/2020 0:00:00 |3318       |
|16/6/2020 0:00:00 |3232       |
|23/6/2020 0:00:00 |3230       |
|11/6/2020 0:00:00 |2747       |
|20/6/2020 0:00:00 |2684       |
|12/6/2020 0:00:00 |2679       |
|10/6/2020 0:00:00 |2650       |
|24/6/2020 0:00:00 |2599       |
+------------------+-----------+

### 4. Distribución de casos por edad (SQL)

In [25]:
sql_edad = spark.sql("""
    SELECT 
        rango_edad,
        COUNT(*) as total_casos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as porcentaje
    FROM covid_edad
    GROUP BY rango_edad
    ORDER BY rango_edad
""")

sql_edad.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----------+----------+
|rango_edad|total_casos|porcentaje|
+----------+-----------+----------+
|0-17      |8587       |8.59      |
|18-29     |24664      |24.66     |
|30-39     |23029      |23.03     |
|40-49     |15828      |15.83     |
|50-59     |12857      |12.86     |
|60-69     |7955       |7.96      |
|70-79     |4362       |4.36      |
|80+       |2718       |2.72      |
+----------+-----------+----------+

### 5. Tasa de letalidad por departamento (SQL)

In [27]:
sql_letalidad = spark.sql("""
    SELECT 
        departamento,
        COUNT(*) as total_casos,
        SUM(CASE WHEN estado = 'FALLECIDO' THEN 1 ELSE 0 END) as fallecidos,
        ROUND(
            SUM(CASE WHEN estado = 'FALLECIDO' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) as tasa_letalidad_porcentaje
    FROM covid
    GROUP BY departamento
    ORDER BY tasa_letalidad_porcentaje DESC
    LIMIT 15
""")

sql_letalidad.show(truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-----------+----------+-------------------------+
|departamento   |total_casos|fallecidos|tasa_letalidad_porcentaje|
+---------------+-----------+----------+-------------------------+
|PUTUMAYO       |28         |7         |25.00                    |
|MAGDALENA      |869        |100       |11.51                    |
|CORDOBA        |807        |89        |11.03                    |
|NORTE SANTANDER|343        |28        |8.16                     |
|SUCRE          |1317       |103       |7.82                     |
|GUAJIRA        |470        |34        |7.23                     |
|GUAINIA        |14         |1         |7.14                     |
|BARRANQUILLA   |13065      |926       |7.09                     |
|STA MARTA D.E. |1022       |69        |6.75                     |
|ATLANTICO      |10994      |662       |6.02                     |
|BOLIVAR        |998        |60        |6.01                     |
|CAUCA          |402        |23        |5.72                  

## Guardar resultados en S3

In [32]:
output_bucket = "lmtorresv-datalake"
output_base = f"s3a://{output_bucket}/covid/preguntas-negocio/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [37]:
def guardar_resultado(dataframe, nombre):
    dataframe.coalesce(1) \
        .write.mode("overwrite") \
        .option("header", True) \
        .csv(output_base + "csv/" + nombre)
    dataframe.coalesce(1) \
        .write.mode("overwrite") \
        .parquet(output_base + "parquet/" + nombre)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### Guardar resultados de DataFrames

In [38]:
guardar_resultado(top_departamentos, "top_10_departamentos")
guardar_resultado(top_ciudades, "top_10_ciudades")
guardar_resultado(top_dias, "top_10_dias")
guardar_resultado(distribucion_edad, "distribucion_por_edad")
guardar_resultado(letalidad_departamento, "letalidad_por_departamento")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### Guardar resultados de SparkSQL

In [39]:
guardar_resultado(sql_departamentos, "sql_top_10_departamentos")
guardar_resultado(sql_ciudades, "sql_top_10_ciudades")
guardar_resultado(sql_dias, "sql_top_10_dias")
guardar_resultado(sql_edad, "sql_distribucion_por_edad")
guardar_resultado(sql_letalidad, "sql_letalidad_por_departamento")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Limpieza

In [36]:
df.unpersist()
df_edad.unpersist()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[fecha_reporte: string, id_caso: int, fecha_notificacion: string, cod_depto: int, departamento: string, cod_municipio: int, ciudad: string, edad: int, unidad_edad: int, sexo: string, tipo_contagio: string, ubicacion: string, estado: string, cod_pais: int, pais: string, recuperado: string, fecha_sintomas: string, fecha_muerte: string, fecha_diagnostico: string, fecha_recuperacion: string, tipo_recuperacion: string, etnia: int, grupo_etnico: string, rango_edad: string]